In [1]:
#Emisor código de Hamming

def posiciones_paridad(n):
    i = 1
    pos = []
    while i <= n:
        pos.append(i)
        i *= 2
    return pos

def hamming_codificar(mensaje):
    m = len(mensaje)
    
    r = 0
    while (2 ** r) < (m + r + 1):
        r += 1

    n = m + r
    par_pos = posiciones_paridad(n)

    codigo = [0] * (n + 1)  
    data_bits = [int(c) for c in mensaje]
    msg_index = 0
    pos = 1
    data_pos_map = {}  

    for i in range(1, n + 1):
        if i not in par_pos:
            codigo[i] = data_bits[msg_index]
            data_pos_map[pos] = i
            pos += 1
            msg_index += 1

    for p in par_pos:
        suma = 0
        for logical_idx in range(1, m + 1):
            if logical_idx & p:
                codigo[p] += codigo[data_pos_map[logical_idx]]
        codigo[p] = codigo[p] % 2

    return ''.join(str(codigo[i]) for i in range(1, n + 1))



In [3]:
# Emisor código Fletcher Checksum

def dividir_en_bloques(data_binaria, bloque_bits):
    padding = (-len(data_binaria)) % bloque_bits
    data_binaria += '0' * padding

    bloques = [data_binaria[i:i + bloque_bits] for i in range(0, len(data_binaria), bloque_bits)]
    return bloques

def fletcher_checksum_bloques(data_binaria, bloque_bits=8):

    if bloque_bits not in [8, 16, 32]:
        raise ValueError("El tamaño de bloque debe ser 8, 16 o 32 bits")

    bloques = dividir_en_bloques(data_binaria, bloque_bits)
    sum1 = 0
    sum2 = 0
    mod = 2 ** bloque_bits - 1 

    for bloque in bloques:
        valor = int(bloque, 2)
        sum1 = (sum1 + valor) % mod
        sum2 = (sum2 + sum1) % mod

    checksum1 = format(sum1, f'0{bloque_bits}b')
    checksum2 = format(sum2, f'0{bloque_bits}b')

    checksum_total = checksum1 + checksum2
    mensaje_codificado = data_binaria + checksum_total

    return {
        "mensaje_original": data_binaria,
        "checksum": checksum_total,
        "mensaje_codificado": mensaje_codificado
    }


In [17]:
import random

def ruido(cadena, probabilidad=0.001):
    lista = list(cadena)
    for i in range(len(lista)):
        if random.random() < probabilidad:
            lista[i] = '1' if lista[i] == '0' else '0'
    return ''.join(lista)


In [18]:
mensaje = input("Ingresa un carácter: ")

if len(mensaje) != 1:
    print("Por favor, ingresa solo un carácter.")
else:
    ascii_valor = ord(mensaje)
    binario_ascii = format(ascii_valor, '08b')


    hamming = hamming_codificar(binario_ascii)
    fletcher = fletcher_checksum_bloques(binario_ascii, bloque_bits=32)
    
    hamming_ruido = ruido(hamming, probabilidad=0.001)
    fletcher_ruido = ruido(fletcher['mensaje_codificado'], probabilidad=0.001)

    print(f"\nCarácter: {mensaje}")
    print(f"Código ASCII: {ascii_valor}")
    print(f"Binario ASCII (8 bits): {binario_ascii}")
    print(f"Mensaje codificado con Hamming: {hamming}")
    print(f"Checksum: {fletcher['checksum']}")
    print(f"Mensaje codificado con Fletcher: {fletcher['mensaje_codificado']}")
    print(f"Mensaje codificado con Hamming con ruido: {hamming_ruido}")
    print(f"Mensaje codificado con Fletcher Checksum con ruido: {fletcher_ruido}")


Ingresa un carácter: A

Carácter: A
Código ASCII: 65
Binario ASCII (8 bits): 01000001
Mensaje codificado con Hamming: 010010010001
Checksum: 0100000100000000000000000000000001000001000000000000000000000000
Mensaje codificado con Fletcher: 010000010100000100000000000000000000000001000001000000000000000000000000
Mensaje codificado con Hamming con ruido: 010010010001
Mensaje codificado con Fletcher Checksum con ruido: 010000010100000100000000000000000000000001000001000000000000000000000000
